# Local RAG Indexing

This notebook builds the local FAISS index used by tutorial 9, `t09_local_rag_agent`.

You will:

1. set a PDF URL;
2. download the PDF into the tutorial folder;
3. extract text and split it into small overlapping chunks;
4. embed each chunk with Google's embedding model;
5. persist the vectors in a local FAISS index;
6. reload the index and test retrieval before starting the ADK agent.

## Prerequisites

```bash
uv sync --extra adk --extra notebooks --extra rag
```

Copy `.env.example` to `.env` first. If `MODEL_PROVIDER=vertex`, embeddings use Vertex AI with Application Default Credentials, `GOOGLE_CLOUD_PROJECT`, and `GOOGLE_CLOUD_LOCATION`, just like the LLM tutorials. Otherwise, set `GOOGLE_API_KEY` for the Gemini API. Optionally set `LOCAL_RAG_PDF_URL` to use a different PDF.

In [1]:
from pathlib import Path
import os
import sys


DEFAULT_PDF_URL = (
    "https://sede.agenciatributaria.gob.es/static_files/Sede/Biblioteca/Manual/"
    "Practicos/IRPF/IRPF-2025/ManualRenta2025Parte1_es_es.pdf"
)


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


def load_env_file(env_file: Path) -> None:
    if not env_file.exists():
        print("No .env file found; expecting provider configuration in the environment")
        return

    for line in env_file.read_text().splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")
    print(f"Loaded {env_file.relative_to(REPO_ROOT)}")


REPO_ROOT = find_repo_root(Path.cwd())
ENV_FILE = REPO_ROOT / ".env"
load_env_file(ENV_FILE)

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tutorials.model_config import get_embedding_model

TUTORIAL_DIR = REPO_ROOT / "src" / "tutorials" / "t09_local_rag_agent"
DOWNLOAD_DIR = TUTORIAL_DIR / "downloads"
STORAGE_DIR = TUTORIAL_DIR / "storage"
PDF_URL = os.getenv("LOCAL_RAG_PDF_URL", DEFAULT_PDF_URL)
MAX_PAGES = int(os.getenv("LOCAL_RAG_MAX_PAGES", "25"))

print(f"PDF URL:         {PDF_URL}")
print(f"Download:        {DOWNLOAD_DIR.relative_to(REPO_ROOT)}")
print(f"Index:           {STORAGE_DIR.relative_to(REPO_ROOT)}")
print(f"Max pages:       {MAX_PAGES or 'all'}")
print(f"Model provider:  {os.getenv('MODEL_PROVIDER', 'gemini')}")

Loaded .env
PDF URL:         https://sede.agenciatributaria.gob.es/static_files/Sede/Biblioteca/Manual/Practicos/IRPF/IRPF-2025/ManualRenta2025Parte1_es_es.pdf
Download:        src/tutorials/t09_local_rag_agent/downloads
Index:           src/tutorials/t09_local_rag_agent/storage
Max pages:       all
Model provider:  vertex


## 1. Download The PDF And Create Chunks

Set `LOCAL_RAG_PDF_URL` in `.env` or edit `PDF_URL` above. The notebook downloads the PDF, extracts text, and uses `SentenceSplitter` to create overlapping chunks before indexing.

The default `LOCAL_RAG_MAX_PAGES=25` keeps classroom runs fast and cheaper. Set it to `0` to process the full PDF.

In [2]:
from urllib.parse import unquote, urlparse
from urllib.request import urlretrieve

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter


def filename_from_url(url: str) -> str:
    path = unquote(urlparse(url).path)
    filename = Path(path).name or "source.pdf"
    return filename if filename.lower().endswith(".pdf") else f"{filename}.pdf"


DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = DOWNLOAD_DIR / filename_from_url(PDF_URL)

if not PDF_PATH.exists():
    urlretrieve(PDF_URL, PDF_PATH)

print(f"Using PDF: {PDF_PATH.relative_to(REPO_ROOT)}")
print(f"Size:      {PDF_PATH.stat().st_size / 1024 / 1024:.1f} MB")

documents = SimpleDirectoryReader(input_files=[str(PDF_PATH)]).load_data()
if MAX_PAGES:
    documents = documents[:MAX_PAGES]

splitter = SentenceSplitter(chunk_size=900, chunk_overlap=150)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Loaded {len(documents)} PDF pages/documents")
print(f"Created {len(nodes)} chunks")
print("\nFirst chunk preview:\n")
print(nodes[0].text[:700])

Using PDF: src/tutorials/t09_local_rag_agent/downloads/ManualRenta2025Parte1_es_es.pdf
Size:      7.2 MB
Loaded 1270 PDF pages/documents
Created 1543 chunks

First chunk preview:

Manual práctico de Renta 2025. Parte 1.
Esta publicación tiene efectos meramente informativos.
Índice
Número de identificación de la publicación (NIPO)•
Presentación•
Guía de las principales novedades del IRPF en el ejercicio 2025
Gestión del impuesto•
Exenciones•
Rendimientos de trabajo•
Rendimiento de actividades económicas•
Regímenes especiales•
Base liquidable•
Mínimo personal y familiar•
Cálculo del impuesto: determinación de las cuotas íntegras•
Deducciones de la cuota íntegra•
Deducciones de la cuota líquida total•
Cuota diferencial•
Resultado de la declaración•
Otras cuestiones de interés•
•
Capítulo 1. Campaña de la declaración de Renta 2025•
Manual práctico de Renta 2025. Parte 1.



## 2. Build And Persist The FAISS Index

FAISS is local: it runs in this Python process and writes index files to `storage/`. The only remote call here is the embedding model.

In [3]:
import shutil

import faiss
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.faiss import FaissVectorStore

embed_model = get_embedding_model()
embedding_dim = len(embed_model.get_text_embedding("dimension probe"))

if STORAGE_DIR.exists():
    shutil.rmtree(STORAGE_DIR)

faiss_index = faiss.IndexFlatL2(embedding_dim)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
)
index.storage_context.persist(persist_dir=str(STORAGE_DIR))

print(f"Embedding dimension: {embedding_dim}")
print(f"Persisted local FAISS index to {STORAGE_DIR.relative_to(REPO_ROOT)}")

2026-05-05 22:37:25,602 - INFO - Loading faiss.
2026-05-05 22:37:25,611 - INFO - Successfully loaded faiss.
2026-05-05 22:37:28,231 - INFO - HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/edem-practica-mlops/locations/us-central1/publishers/google/models/text-embedding-004:predict "HTTP/1.1 200 OK"
2026-05-05 22:37:29,176 - INFO - HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/edem-practica-mlops/locations/us-central1/publishers/google/models/text-embedding-004:predict "HTTP/1.1 200 OK"
2026-05-05 22:37:30,100 - INFO - HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/edem-practica-mlops/locations/us-central1/publishers/google/models/text-embedding-004:predict "HTTP/1.1 200 OK"
2026-05-05 22:37:31,146 - INFO - HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/edem-practica-mlops/locations/us-central1/publishers/google/models/text-embedding-004:predict "H

Embedding dimension: 768
Persisted local FAISS index to src/tutorials/t09_local_rag_agent/storage


## 3. Reload And Test Retrieval

This mirrors what the ADK agent will do at startup: load the persisted vector store, create a retriever, and retrieve context for a question.

In [4]:
from llama_index.core import StorageContext, load_index_from_storage

vector_store = FaissVectorStore.from_persist_dir(str(STORAGE_DIR))
storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    persist_dir=str(STORAGE_DIR),
)
loaded_index = load_index_from_storage(
    storage_context=storage_context,
    embed_model=embed_model,
)

retriever = loaded_index.as_retriever(similarity_top_k=3)
query = "Qué novedades principales incluye el manual?"
matches = retriever.retrieve(query)

print(f"Query: {query}\n")
for i, match in enumerate(matches, start=1):
    print(f"--- Match {i} | score={match.score:.4f} ---")
    print(match.node.text[:700].strip())
    print()

2026-05-05 22:39:37,012 - INFO - Loading llama_index.vector_stores.faiss.base from /Users/danielruiz/Documents/GIT/agents-tutorials/src/tutorials/t09_local_rag_agent/storage/default__vector_store.json.
2026-05-05 22:39:37,028 - INFO - Loading all indices.
2026-05-05 22:39:37,277 - INFO - HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/edem-practica-mlops/locations/us-central1/publishers/google/models/text-embedding-004:predict "HTTP/1.1 200 OK"


Query: Qué novedades principales incluye el manual?

--- Match 1 | score=0.8660 ---
Constituye, en cualquier caso y en el marco de la campaña de Renta 2025, una buena 
oportunidad para quienes deseen profundizar en el conocimiento del impuesto.
Por otra parte, su confección en este formato digital HTML persigue alcanzar tres objetivos 
básicos que venían siendo reiteradamente demandados por los contribuyentes.
El primero es su accesibilidad web, es decir, lograr que el acceso al mismo sea posible por el 
máximo número de personas, independientemente de sus conocimientos o capacidades 
personales o físicas e independientemente de las características técnicas del equipo utilizado 
para acceder a la web.
El segundo, posibilitar y poner a disposición de los contribuyentes de fo

--- Match 2 | score=0.8947 ---
Presentación
La Agencia Estatal de Administración Tributaria tiene entre sus principales objetivos el de 
minimizar los costes de cumplimiento que debe soportar la ciudadanía en sus r

## 4. Run The Agent

After the index exists, start ADK Web from the repository root:

```bash
uv run adk web src/tutorials/t09_local_rag_agent
```

Open http://localhost:8000, select **local_rag**, and ask a question about the downloaded PDF.